In [ ]:
import time
import requests
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

LAT = 52.350839
LON = 11.092744673377407

propiedades = ["clay", "sand", "silt", "phh2o", "soc"]
profundidades = ["0-5cm", "5-15cm", "15-30cm"]
pesos = {"0-5cm": 5, "5-15cm": 10, "15-30cm": 15}

# Configuración de sesión robusta con reintentos
session = requests.Session()
reintentos = Retry(
    total=3,
    backoff_factor=2,  # Espera 2s, 4s, 8s entre intentos
    status_forcelist=[429, 500, 502, 503, 504],
)
session.mount("https://", HTTPAdapter(max_retries=reintentos))

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

try:
    respuesta = session.get(
        "https://rest.isric.org/soilgrids/v2.0/properties/query",
        params={
            "lat": LAT,
            "lon": LON,
            "property": propiedades,
            "depth": profundidades,
            "value": "mean",
        },
        headers=headers,
        timeout=120,  # Aumentamos a 120 segundos
    )
    respuesta.raise_for_status()
    datos = respuesta.json()
except requests.exceptions.RequestException as e:
    print(f"No se pudo conectar con el servidor de SoilGrids: {e}")
    datos = None

if datos:
    filas = []
    for capa in datos["properties"]["layers"]:
        unidades = capa["unit_measure"]
        factor = unidades["d_factor"]

        fila = {
            "propiedad": capa["name"],
            "unidad": unidades["target_units"],
        }

        for profundidad in capa["depths"]:
            valor = profundidad["values"].get("mean")
            fila[profundidad["label"]] = (
                valor / factor if valor is not None else None
            )

        if all(fila.get(p) is not None for p in profundidades):
            fila["0-30cm"] = sum(
                fila[p] * pesos[p] for p in profundidades
            ) / 30
        else:
            fila["0-30cm"] = None

        filas.append(fila)

    tabla = pd.DataFrame(filas)
    display(tabla.round(2))

In [ ]:
import json
from google.colab import files
from shapely.geometry import shape

subidos = files.upload()

nombre = next(
    nombre for nombre in subidos
    if nombre.lower().endswith(".geojson")
)

geojson = json.loads(subidos[nombre])

parcelas = [
    parcela
    for parcela in geojson["features"]
    if parcela["properties"].get("isArchived") is False
]

print("Parcelas no archivadas:", len(parcelas))

NOMBRE_PARCELA = "Mittelbreite"

parcela = next(
    parcela
    for parcela in parcelas
    if parcela["properties"]["fieldName"] == NOMBRE_PARCELA
)

geometria = shape(parcela["geometry"])

if geometria.is_empty or not geometria.is_valid:
    raise ValueError("La geometría de la parcela está vacía o no es válida.")

punto = geometria.representative_point()

LON = punto.x
LAT = punto.y

print("Parcela:", NOMBRE_PARCELA)
print("Identificador:", parcela["properties"]["plotId"])
print("Latitud:", LAT)
print("Longitud:", LON)

In [ ]:
import requests
import pandas as pd

propiedades = ["clay", "sand", "silt", "phh2o", "soc"]
profundidades = ["0-5cm", "5-15cm", "15-30cm"]
estadisticas = ["mean", "Q0.05", "Q0.5", "Q0.95"]

respuesta = requests.get(
    "https://rest.isric.org/soilgrids/v2.0/properties/query",
    params={
        "lat": LAT,
        "lon": LON,
        "property": propiedades,
        "depth": profundidades,
        "value": estadisticas,
    },
    timeout=90,
)
respuesta.raise_for_status()
datos = respuesta.json()

capas = datos.get("properties", {}).get("layers", [])

if not capas:
    raise ValueError("La respuesta no contiene capas de suelo.")

filas = []

for capa in capas:
    unidades = capa["unit_measure"]
    factor = unidades["d_factor"]

    for profundidad in capa["depths"]:
        fila = {
            "parcela": NOMBRE_PARCELA,
            "propiedad": capa["name"],
            "profundidad": profundidad["label"],
            "unidad": unidades["target_units"],
        }

        for estadistica in estadisticas:
            valor = profundidad["values"].get(estadistica)
            fila[estadistica] = (
                valor / factor if valor is not None else None
            )

        filas.append(fila)

tabla = pd.DataFrame(filas)
tabla["amplitud_intervalo"] = tabla["Q0.95"] - tabla["Q0.05"]

display(tabla.round(2))

Hasta ahora solo hemos obtenido los datos en 3 capas diferentes, pero solo de un punto de la parcela.
Vamos a pasar a toda la parcela. PERO SOLO VAMOS A BUSCAR EL PARAMETRO ARCILLA

In [ ]:
%pip -q install rasterio shapely pyproj #NO TOCAR

Selección de la parcela que nos interesa

In [ ]:
import json
from google.colab import files
from shapely.geometry import shape

if "geojson" not in globals():
    subidos = files.upload()
    nombre = next(
        n for n in subidos
        if n.lower().endswith(".geojson")
    )
    geojson = json.loads(subidos[nombre])

NOMBRE_PARCELA = "Mittelbreite"

parcela = next(
    f for f in geojson["features"]
    if f["properties"]["fieldName"] == NOMBRE_PARCELA
    and f["properties"].get("isArchived") is False
)

geometria = shape(parcela["geometry"])

if geometria.is_empty or not geometria.is_valid:
    raise ValueError("La geometría está vacía o no es válida.")

print("Parcela seleccionada:", NOMBRE_PARCELA)

Lectura de píxeles y calculo de la media de los datos de la parcela (Este código solicita ventanas pequeña del raster alrededor de la parcela)

In [ ]:
import math
import numpy as np
import pandas as pd
import rasterio

from pyproj import Transformer
from shapely.geometry import Polygon
from shapely.ops import transform
from rasterio.windows import Window, from_bounds

url = (
    "https://files.isric.org/soilgrids/latest/data/"
    "clay/clay_0-5cm_mean.vrt"
)

filas = []

with rasterio.Env(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    GDAL_HTTP_CONNECTTIMEOUT="15",
    GDAL_HTTP_TIMEOUT="90",
    GDAL_HTTP_MAX_RETRY="2",
):
    with rasterio.open("/vsicurl/" + url) as raster:
        # Llevar la parcela al sistema de coordenadas del raster.
        conversor = Transformer.from_crs(
            "EPSG:4326", raster.crs, always_xy=True
        )
        campo = transform(conversor.transform, geometria)

        # Ventana que incluye todos los píxeles del contorno.
        ventana_base = from_bounds(
            *campo.bounds, transform=raster.transform
        )

        col0 = math.floor(ventana_base.col_off)
        fila0 = math.floor(ventana_base.row_off)
        col1 = math.ceil(ventana_base.col_off + ventana_base.width)
        fila1 = math.ceil(ventana_base.row_off + ventana_base.height)

        ventana = Window(
            col0, fila0, col1 - col0, fila1 - fila0
        ).intersection(Window(0, 0, raster.width, raster.height))

        datos = raster.read(1, window=ventana, masked=True)
        rejilla = raster.window_transform(ventana)

        print("Tamaño del píxel en metros:", raster.res)

        for fila in range(datos.shape[0]):
            for columna in range(datos.shape[1]):
                pixel = Polygon([
                    rejilla * (columna, fila),
                    rejilla * (columna + 1, fila),
                    rejilla * (columna + 1, fila + 1),
                    rejilla * (columna, fila + 1),
                ])

                superficie = campo.intersection(pixel).area

                if superficie <= 0:
                    continue

                valor = datos[fila, columna]
                valido = (
                    not np.ma.is_masked(valor)
                    and np.isfinite(float(valor))
                )

                filas.append({
                    "fila_raster": int(ventana.row_off) + fila,
                    "columna_raster": int(ventana.col_off) + columna,
                    "superficie_m2": superficie,
                    "arcilla_pct": float(valor) / 10 if valido else np.nan,
                })

pixeles = pd.DataFrame(filas)

if pixeles.empty:
    raise ValueError("No se encontraron píxeles dentro de la parcela.")

validos = pixeles.dropna(subset=["arcilla_pct"]).copy()

if validos.empty:
    raise ValueError("Los píxeles de la parcela no tienen datos.")

area_valida = validos["superficie_m2"].sum()
cobertura = 100 * area_valida / campo.area

media = np.average(
    validos["arcilla_pct"],
    weights=validos["superficie_m2"],
)

validos["peso_pct"] = (
    100 * validos["superficie_m2"] / area_valida
)

display(validos.round(2))

print(f"Píxeles que intersectan la parcela: {len(pixeles)}")
print(f"Píxeles con datos: {len(validos)}")
print(f"Superficie de la parcela con datos: {cobertura:.2f} %")
print(f"Arcilla media en 0–5 cm: {media:.2f} %")

Vamos a hacer lo mismo para las 5 variables y para los primeros 30 cm. (Solo usaré pixeles con datos en las 3 profundidades)

Lectura de las 5 propiedades y 3 profundidades

In [ ]:
import numpy as np
import pandas as pd
import rasterio

from pyproj import Transformer
from shapely.ops import transform

propiedades = {
    "clay": ("Arcilla", "%"),
    "sand": ("Arena", "%"),
    "silt": ("Limo", "%"),
    "phh2o": ("pH en agua", "pH"),
    "soc": ("Carbono orgánico", "g/kg"),
}

profundidades = ["0-5cm", "5-15cm", "15-30cm"]
espesores = np.array([5, 10, 15])

# Estas cinco propiedades usan un divisor de 10
# en los archivos originales de SoilGrids.
factor = 10

if "cache_rasters" not in globals():
    cache_rasters = {}

base = pixeles[
    ["fila_raster", "columna_raster", "superficie_m2"]
].copy().reset_index(drop=True)

filas_locales = (
    base["fila_raster"].to_numpy() - int(ventana.row_off)
).astype(int)

columnas_locales = (
    base["columna_raster"].to_numpy() - int(ventana.col_off)
).astype(int)

# Identifica la parcela y la ventana de esta ejecución.
clave_zona = (
    geometria.wkb_hex,
    tuple(ventana.flatten()),
    tuple(rejilla),
    tuple(zip(base["fila_raster"], base["columna_raster"])),
)

lecturas = {}
errores = []

with rasterio.Env(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    GDAL_HTTP_CONNECTTIMEOUT="15",
    GDAL_HTTP_TIMEOUT="90",
    GDAL_HTTP_MAX_RETRY="2",
):
    for propiedad in propiedades:
        for profundidad in profundidades:
            url = (
                "https://files.isric.org/soilgrids/latest/data/"
                f"{propiedad}/{propiedad}_{profundidad}_mean.vrt"
            )

            clave = (clave_zona, url)

            if clave in cache_rasters:
                print(f"En memoria: {propiedad} {profundidad}")
                lecturas[(propiedad, profundidad)] = cache_rasters[clave]
                continue

            print(f"Leyendo: {propiedad} {profundidad}")

            try:
                with rasterio.open("/vsicurl/" + url) as raster:
                    # Comprobar que podemos reutilizar la misma rejilla.
                    if not raster.window_transform(ventana).almost_equals(rejilla):
                        raise ValueError("La rejilla no coincide con la inicial.")

                    conversor = Transformer.from_crs(
                        "EPSG:4326", raster.crs, always_xy=True
                    )
                    campo_actual = transform(
                        conversor.transform, geometria
                    )

                    if not campo_actual.equals_exact(campo, tolerance=0.01):
                        raise ValueError(
                            "El sistema de coordenadas no coincide."
                        )

                    bloque = raster.read(
                        1, window=ventana, masked=True
                    ).astype("float64").filled(np.nan)

                    valores = (
                        bloque[filas_locales, columnas_locales] / factor
                    )
                    valores[~np.isfinite(valores)] = np.nan

                cache_rasters[clave] = valores
                lecturas[(propiedad, profundidad)] = valores

            except Exception as error:
                errores.append({
                    "propiedad": propiedad,
                    "profundidad": profundidad,
                    "error": str(error),
                })
                print(f"  Error: {error}")

if errores:
    display(pd.DataFrame(errores))
else:
    print("Las 15 capas se han leído correctamente.")

In [ ]:
from rasterio.transform import array_bounds
from rasterio.windows import from_bounds

# Límites geográficos de la ventana original.
limites = array_bounds(
    int(ventana.height),
    int(ventana.width),
    rejilla,
)

with rasterio.Env(
    GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
    GDAL_HTTP_CONNECTTIMEOUT="15",
    GDAL_HTTP_TIMEOUT="90",
    GDAL_HTTP_MAX_RETRY="2",
):
    for profundidad in profundidades:
        url = (
            "https://files.isric.org/soilgrids/latest/data/"
            f"soc/soc_{profundidad}_mean.vrt"
        )

        print(f"Leyendo carbono: {profundidad}")

        with rasterio.open("/vsicurl/" + url) as raster:
            conversor = Transformer.from_crs(
                "EPSG:4326", raster.crs, always_xy=True
            )
            campo_actual = transform(conversor.transform, geometria)

            if not campo_actual.equals_exact(campo, tolerance=0.01):
                raise ValueError("El sistema de coordenadas es distinto.")

            # Misma zona geográfica, distinta numeración de columnas.
            ventana_soc = from_bounds(
                *limites,
                transform=raster.transform,
            ).round_offsets().round_lengths()

            if not raster.window_transform(ventana_soc).almost_equals(rejilla):
                raise ValueError("Los píxeles no quedan alineados.")

            bloque = raster.read(
                1,
                window=ventana_soc,
                masked=True,
            ).astype("float64").filled(np.nan)

            if bloque.shape != (
                int(ventana.height),
                int(ventana.width),
            ):
                raise ValueError("El tamaño de la ventana no coincide.")

            valores = (
                bloque[filas_locales, columnas_locales] / 10
            )
            valores[~np.isfinite(valores)] = np.nan

        lecturas[("soc", profundidad)] = valores
        cache_rasters[(clave_zona, url)] = valores

        print("  Correcto")

print("Carbono incorporado. Vuelve a ejecutar la celda del resumen.")

Calculo del resumen de toda la parcela

In [ ]:
resumen = []
detalle_pixeles = {}

for propiedad, (nombre, unidad) in propiedades.items():
    resultado = {
        "propiedad": nombre,
        "unidad": unidad,
        "media_0_30cm": np.nan,
        "min_pixel_0_30cm": np.nan,
        "max_pixel_0_30cm": np.nan,
        "pixeles_validos": 0,
        "cobertura_pct": np.nan,
        "estado": "error de lectura",
    }

    if not all(
        (propiedad, profundidad) in lecturas
        for profundidad in profundidades
    ):
        resumen.append(resultado)
        continue

    # Una fila por píxel y una columna por profundidad.
    matriz = np.column_stack([
        lecturas[(propiedad, profundidad)]
        for profundidad in profundidades
    ])

    completos = np.isfinite(matriz).all(axis=1)
    valores_0_30 = np.full(len(base), np.nan)

    # Promedio por espesor dentro de cada píxel.
    valores_0_30[completos] = (
        matriz[completos] @ espesores
    ) / espesores.sum()

    detalle = base.copy()

    for indice, profundidad in enumerate(profundidades):
        detalle[profundidad] = matriz[:, indice]

    detalle["0-30cm"] = valores_0_30
    detalle_pixeles[propiedad] = detalle

    area_valida = base.loc[completos, "superficie_m2"].sum()

    resultado["pixeles_validos"] = int(completos.sum())
    resultado["cobertura_pct"] = 100 * area_valida / campo.area
    resultado["estado"] = "sin datos completos"

    if completos.any():
        valores = valores_0_30[completos]
        superficies = base.loc[completos, "superficie_m2"]

        resultado["media_0_30cm"] = np.average(
            valores, weights=superficies
        )
        resultado["min_pixel_0_30cm"] = valores.min()
        resultado["max_pixel_0_30cm"] = valores.max()
        resultado["estado"] = "ok"

    resumen.append(resultado)

tabla_parcela = pd.DataFrame(resumen)

print("Parcela:", NOMBRE_PARCELA)
print("Fuente: ISRIC SoilGrids")
print("Profundidad: 0–30 cm")
display(tabla_parcela.round(2))

Dibujo de los resultados

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from rasterio.transform import array_bounds

configuracion = {
    "clay": ("Arcilla", "%", "YlOrBr"),
    "sand": ("Arena", "%", "YlOrBr"),
    "silt": ("Limo", "%", "YlOrBr"),
    "phh2o": ("pH en agua", "pH", "viridis"),
    "soc": ("Carbono orgánico", "g/kg", "YlGn"),
}

alto = int(ventana.height)
ancho = int(ventana.width)

oeste, sur, este, norte = array_bounds(alto, ancho, rejilla)

# Coordenadas locales en metros para facilitar la lectura.
extension = [0, este - oeste, 0, norte - sur]

poligonos = (
    [campo]
    if campo.geom_type == "Polygon"
    else list(campo.geoms)
)

fig, ejes = plt.subplots(2, 3, figsize=(16, 10))
ejes = ejes.ravel()

for eje, (codigo, (nombre, unidad, colores)) in zip(
    ejes, configuracion.items()
):
    imagen = np.full((alto, ancho), np.nan)
    detalle = detalle_pixeles[codigo]

    for _, pixel in detalle.iterrows():
        fila = int(pixel["fila_raster"] - ventana.row_off)
        columna = int(pixel["columna_raster"] - ventana.col_off)
        imagen[fila, columna] = pixel["0-30cm"]

    mapa = eje.imshow(
        np.ma.masked_invalid(imagen),
        extent=extension,
        origin="upper",
        interpolation="nearest",
        cmap=colores,
    )

    # Dibujar el contorno y los posibles huecos de la parcela.
    for poligono in poligonos:
        for anillo in [poligono.exterior, *poligono.interiors]:
            x, y = anillo.xy
            eje.plot(
                np.asarray(x) - oeste,
                np.asarray(y) - sur,
                color="black",
                linewidth=1.5,
            )

    # Mostrar los límites de los píxeles originales.
    eje.set_xticks(
        np.arange(ancho + 1) * abs(rejilla.a),
        minor=True,
    )
    eje.set_yticks(
        np.arange(alto + 1) * abs(rejilla.e),
        minor=True,
    )
    eje.grid(which="minor", color="white", linewidth=0.6)

    eje.set_title(f"{nombre} · 0–30 cm")
    eje.set_xlabel("Distancia horizontal (m)")
    eje.set_ylabel("Distancia vertical (m)")
    eje.set_aspect("equal")

    fig.colorbar(mapa, ax=eje, label=unidad, shrink=0.8)

ejes[-1].axis("off")

fig.suptitle(
    f"{NOMBRE_PARCELA} — Predicciones de ISRIC SoilGrids",
    fontsize=16,
)

plt.tight_layout()
plt.show()

Función para leer cualquier capa

In [ ]:
from rasterio.transform import array_bounds
from rasterio.windows import from_bounds

def leer_capa(propiedad, profundidad, estadistica):
    url = (
        "https://files.isric.org/soilgrids/latest/data/"
        f"{propiedad}/{propiedad}_{profundidad}_{estadistica}.vrt"
    )

    clave = (clave_zona, url)

    if clave in cache_rasters:
        return cache_rasters[clave].copy()

    limites = array_bounds(
        int(ventana.height),
        int(ventana.width),
        rejilla,
    )

    with rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        GDAL_HTTP_CONNECTTIMEOUT="15",
        GDAL_HTTP_TIMEOUT="90",
        GDAL_HTTP_MAX_RETRY="2",
    ):
        with rasterio.open("/vsicurl/" + url) as raster:
            conversor = Transformer.from_crs(
                "EPSG:4326", raster.crs, always_xy=True
            )
            campo_actual = transform(conversor.transform, geometria)

            if not campo_actual.equals_exact(campo, tolerance=0.01):
                raise ValueError("Sistema de coordenadas diferente.")

            ventana_capa = from_bounds(
                *limites,
                transform=raster.transform,
            ).round_offsets().round_lengths()

            if not raster.window_transform(ventana_capa).almost_equals(rejilla):
                raise ValueError("Los píxeles no están alineados.")

            bloque = raster.read(
                1,
                window=ventana_capa,
                masked=True,
            ).astype("float64").filled(np.nan)

            if bloque.shape != (
                int(ventana.height),
                int(ventana.width),
            ):
                raise ValueError("Tamaño de ventana diferente.")

            # Válido para clay, sand, silt, phh2o y soc.
            valores = bloque[filas_locales, columnas_locales] / 10
            valores[~np.isfinite(valores)] = np.nan

    cache_rasters[clave] = valores.copy()
    return valores

Obtener el intervalo y resumir apmplitud: Esta celda necesita 30 capas nuevas: dos percentiles * tres profundidades * cinco propiedades.

In [ ]:
incertidumbre_pixeles = {}
resumen_incertidumbre = []

for propiedad, (nombre, unidad) in propiedades.items():
    for profundidad in profundidades:
        print(f"Procesando {nombre}: {profundidad}", flush=True)

        resultado = {
            "propiedad": nombre,
            "profundidad": profundidad,
            "unidad_amplitud": (
                "puntos porcentuales" if unidad == "%" else unidad
            ),
            "amplitud_media": np.nan,
            "pixeles_validos": 0,
            "cobertura_pct": np.nan,
            "estado": "error",
            "error": None,
        }

        try:
            q05 = leer_capa(propiedad, profundidad, "Q0.05")
            q95 = leer_capa(propiedad, profundidad, "Q0.95")

            validos = np.isfinite(q05) & np.isfinite(q95)

            if np.any(q95[validos] < q05[validos]):
                raise ValueError("Hay percentiles en orden incorrecto.")

            detalle = base.copy()
            detalle["q05"] = q05
            detalle["q95"] = q95
            detalle["amplitud"] = np.where(
                validos, q95 - q05, np.nan
            )

            incertidumbre_pixeles[
                (propiedad, profundidad)
            ] = detalle

            superficies = base.loc[validos, "superficie_m2"]
            area_valida = superficies.sum()

            resultado["pixeles_validos"] = int(validos.sum())
            resultado["cobertura_pct"] = (
                100 * area_valida / campo.area
            )

            if validos.any():
                resultado["amplitud_media"] = np.average(
                    q95[validos] - q05[validos],
                    weights=superficies,
                )
                resultado["estado"] = "ok"
            else:
                resultado["estado"] = "sin datos"

        except Exception as error:
            resultado["error"] = str(error)
            print(f"  Error: {error}", flush=True)

        resumen_incertidumbre.append(resultado)

tabla_incertidumbre = pd.DataFrame(resumen_incertidumbre)

display(tabla_incertidumbre.round(2))

¿Qué significa el resultado?
Si para arcilla en 0-5 cm obtuvieramos amplitud media de 25 puntos porcentuales, significaría que los intervalos de los píxeles tienen, en promedio ponderado por superficie, es anchura. No significaría "arcilla de la parcela +-25 %"

Para ver los intervalos origunales de cada pixel

In [ ]:
display(
    incertidumbre_pixeles[("clay", "0-5cm")].round(2)
)

Calcular nfk

In [ ]:
from google.colab import files

files.upload()  # Selecciona nfk_soilgrids.py

In [ ]:
from nfk_soilgrids import calcular_nfk_parcela

# Conserva las lecturas correctas al volver a ejecutar.
if "cache_agua" not in globals():
    cache_agua = {}

tabla_nfk, detalle_nfk = calcular_nfk_parcela(
    base=base,
    ventana=ventana,
    rejilla=rejilla,
    campo=campo,
    nombre_parcela=NOMBRE_PARCELA,
    fc_kpa=33,
    cache=cache_agua,
)

display(tabla_nfk.round(2))
display(detalle_nfk.round(2))